# Eight-colour fused cyclotomic construction over $\mathbb F_{449}$

Revision of Richard Pennifold's Alm-401 notebook by Charles Gretton 27-Aug-2026. Intention is to demonstrate construction of Ramsey colouring with 8 colours via $\mathbb F_{449}$. Recipe is a fairly general generate and test method with a little combinatorial optimisation (max clique) at the end. Author notes this construction was intended to find a starting point for a broad combinatorial search, but seems to have found a solution to the problem. 

We have found that 8 cyclotomic classes do not seem to work. 

Here, I make 16 classes and then fuse pairs. I look for fused-pair compatibility checking conditions that were documented in Alm's paper, create a graph with pairwise compatibility, and use max clique to extract a candidate solution -- i.e., clique of mutually compatible pairs of size 8, and thereby 8 colours. 

Note, the max clique idea occurred to me on the 27th, and was not used to construct the example circulated on the 26th. 

Solution is checked. 

The notebook:

1. constructs the 16 classes $C_0,\ldots,C_{15}$;

2. forms every pair-union $D_{a,b}=C_a\cup C_b$---i.e., pairwise fusions---keeping only $-D_{a,b}=D_{a,b}$ -- i.e., Ramsey target is an undirected graph;
3. retains candidates satisfying inverse closure $-D=D$ (from above) and  $D_{a,b}+D_{a,b}=\mathbb F_P\setminus D_{a,b} \; (\mathrm{S1})$ -- i.e. $\mathrm{S}$(trong condition) 1;

4. makes a compatibility graph, where adjacency means disjoint fine indices $|\{a,b,c,d\}|=4$ (equivalently $\{a,b\}\cap\{c,d\}=\emptyset$) and satisfaction of $D_{a,b}+D_{c,d}=\mathbb F_P\setminus\{0\} ; \mathrm{S2}$ -- i.e., latter is $\mathrm{S}$(trong condition) 2.


5. asks SageMath/Cliquer for a maximum clique;

6. verifies that a maximum clique has size 8 and therefore gives the required eight colours;

7. constructs and exports the edge-coloured $K_{449}$.

Note: Pairwise compatibility test includes disjointness. Any 8-clique uses $8\times2=16$ distinct fine indices and thus partitions $\mathbb Z_{16}$.


In [1]:
from sage.all import *
from sage.graphs.cliquer import all_cliques

P = 449      # this is one I prepared earlier
R = 16       # number of fine cyclotomic classes
M = 8        # number of final colours


# SAFETY - double check
assert is_prime(P)
assert (P - 1) % (2 * R) == 0   # ensures 16 classes and that $-1\in H$
assert R == 2 * M


## Construct the fine (i.e., not coarse) cyclotomic classes

Let $g$ be a primitive root modulo $P$ and let

$$H=\langle g^{16}\rangle.$$

It follows directly from the definition of a generated cyclic subgroup that the fine classes are

$$C_a=g^aH=\{g^{a+16t}:0\le t<(P-1)/16\}.$$


In [2]:
def sumset(A,B,p):
    return Set((a + b) % p for a in A for b in B)

def build_fine_classes(p,r):
    g = primitive_root(p)
    class_size = (p - 1) // r

    C = [
        Set(power_mod(g,a + r*t,p) for t in range(class_size))
        for a in range(r)
    ]

    nonzero = Set(range(1,p))
    covered = Set(x for X in C for x in X)

    # SAFETY
    assert covered == nonzero
    assert sum(len(X) for X in C) == p - 1
    assert all(len(X) == class_size for X in C)

    return g,C

# Primitive root g and classes C
g,C = build_fine_classes(P,R)
print("Primitive root:",         g)
print("Size of a class:",        len(C[0]))
print("Number of fine classes:", len(C))


Primitive root: 3
Size of a class: 28
Number of fine classes: 16


## Build candidate pair-fusions

A candidate vertex is an unordered pair $(a,b)$ representing

$$D_{a,b}=C_a\cup C_b.$$

A candidate is retained only if it is closed under negation and satisfies

$$(\mathrm{S1})\qquad D_{a,b}+D_{a,b}=\mathbb F_P\setminus D_{a,b}.$$


In [3]:
FULL = Set(range(P))
NONZERO = Set(range(1,P))

candidates = {}
for a in range(R):
    for b in range(a + 1,R):
        D = C[a].union(C[b])
        if Set((-x) % P for x in D) != D:
            continue
        if sumset(D,D,P) != FULL.difference(D):
            continue
        candidates[(a,b)] = D

print("Pairs:",            binomial(R, 2))
print("Surviving pairs:",  len(candidates))


Pairs: 120
Surviving pairs: 16


## Compatibility graph and maximum clique

The vertices are the surviving pair-fusions. Two vertices $(a,b)$ and $(c,d)$ are adjacent exactly when:

1. $\{a,b\}\cap\{c,d\}=\varnothing$; and
2. their fused sets satisfy

$$(\mathrm{S2})\qquad D_{a,b}+D_{c,d}=\mathbb F_P\setminus\{0\}.$$

Thus a clique is a family of pairwise compatible colours. Since every compatibility edge also enforces disjointness, a clique can have size at most 8. A clique of size 8 automatically uses all 16 indices exactly once.


In [4]:
compatibility_graph = Graph()
compatibility_graph.add_vertices(list(candidates))

labels = list(candidates)
for u_pos in range(len(labels)):
    u = labels[u_pos]
    Du = candidates[u]
    for v_pos in range(u_pos + 1,len(labels)):
        v = labels[v_pos]
        if set(u).intersection(v):
            continue
        if sumset(Du,candidates[v], P) == NONZERO:
            compatibility_graph.add_edge(u,v)

print("Compatibility Graph V:", compatibility_graph.order())
print("Compatibility Graph E:", compatibility_graph.size())

# One max clique is all we need -- NOTE: c.g. to investigate better option for performance 
max_clique = next(all_cliques(compatibility_graph))
max_clique = sorted(tuple(pair) for pair in max_clique)

print("Max clique size:", len(max_clique))
print("Max clique:",      max_clique)

if len(max_clique) < M:
    raise ValueError("awful news!")

# Graph sanity
assert len(max_clique) == M
assert sorted(i for pair in max_clique for i in pair) == list(range(R))


Compatibility Graph V: 16
Compatibility Graph E: 104
Max clique size: 8
Max clique: [(0, 10), (1, 7), (2, 8), (3, 13), (4, 14), (5, 11), (6, 12), (9, 15)]


## Double check


In [5]:
selected_pairs = max_clique
D = [candidates[pair] for pair in selected_pairs]

# Combine all eight fused sets into one set.
covered = Set(x for Di in D for x in Di)

# Verify that the fused sets partition all nonzero field elements.
assert covered == NONZERO
assert sum(len(Di) for Di in D) == P - 1
assert all(len(Di) == (P - 1) // M for Di in D)

# Independently verify inverse closure, S1, and S2.
for i in range(M):
    assert Set((-x) % P for x in D[i]) == D[i]
    assert sumset(D[i], D[i], P) == FULL.difference(D[i])

    for j in range(M):
        if i != j:
            assert sumset(D[i], D[j], P) == NONZERO

print("Verified inverse closure, S1, and S2 for all eight colours.")
print("Selected pair-fusions:")

for colour, pair in enumerate(selected_pairs):
    print(
        "  colour %d: C_%d union C_%d"
        % (colour, pair[0], pair[1])
    )

Verified inverse closure, S1, and S2 for all eight colours.
Selected pair-fusions:
  colour 0: C_0 union C_10
  colour 1: C_1 union C_7
  colour 2: C_2 union C_8
  colour 3: C_3 union C_13
  colour 4: C_4 union C_14
  colour 5: C_5 union C_11
  colour 6: C_6 union C_12
  colour 7: C_9 union C_15


## Give me the graph


In [6]:
# Map every nonzero difference to its colour.
difference_colour = {}
for colour, Di in enumerate(D):
    for d in Di:
        assert d not in difference_colour
        difference_colour[Integer(d)] = colour

assert set(difference_colour) == set(range(1, P))

G = graphs.CompleteGraph(P)
for x in range(P - 1):
    for y in range(x + 1, P):
        d = Integer((y - x) % P)
        G.set_edge_label(x, y, difference_colour[d])

assert G.size() == P * (P - 1) // 2
print("constructed K_%d with %d labelled edges" % (P, G.size()))


constructed K_449 with 100576 labelled edges


## Export the graph


In [7]:
pajek_path = "Fusion8_P449.pajek"

G.export_to_file(pajek_path)
print("wrote", pajek_path)


wrote Fusion8_P449.pajek
